# Expanded Results

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
results = pd.read_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/analysis_results/Zombies_expanded_results_for_dir_ols_on_windows.pkl')

In [15]:
results

,NeuronID,WindowStart_ms,WindowEnd_ms,Behavior,Source_Monkey,StimulusMonkey,MeanSpikeRate,R-squared,coef,intercept,p_value
0,AMG_2023-09-26_1_Channel.C_018_Unit 1,50.0,150.0,AffiliationTo,7124,69X,3.333333,0.645158,-0.516683,1.697674,0.016349
1,AMG_2023-09-26_1_Channel.C_018_Unit 1,50.0,150.0,AffiliationTo,7124,72X,5.000000,0.645158,-0.516683,1.697674,0.016349
2,AMG_2023-09-26_1_Channel.C_018_Unit 1,50.0,150.0,AffiliationTo,7124,94B,4.285714,0.645158,-0.516683,1.697674,0.016349
3,AMG_2023-09-26_1_Channel.C_018_Unit 1,50.0,150.0,AffiliationTo,7124,110E,3.333333,0.645158,-0.516683,1.697674,0.016349
4,AMG_2023-09-26_1_Channel.C_018_Unit 1,50.0,150.0,AffiliationTo,7124,67G,0.000000,0.645158,-0.516683,1.697674,0.016349
...,...,...,...,...,...,...,...,...,...,...,...
11483,Unknown_2023-12-18_3_Channel.C_021_Unit 1,600.0,800.0,AgonismFrom,87J,94B,4.000000,0.794198,0.557139,-1.497312,0.002965
11484,Unknown_2023-12-18_3_Channel.C_021_Unit 1,600.0,800.0,AgonismFrom,87J,110E,4.000000,0.794198,0.557139,-1.497312,0.002965
11485,Unknown_2023-12-18_3_Channel.C_021_Unit 1,600.0,800.0,AgonismFrom,87J,67G,5.500000,0.794198,0.557139,-1.497312,0.002965
11486,Unknown_2023-12-18_3_Channel.C_021_Unit 1,600.0,800.0,AgonismFrom,87J,143H,1.000000,0.794198,0.557139,-1.497312,0.002965


In [16]:
neuron = "AMG_2023-09-26_1_Channel.C_014_Unit 1"
neuron_of_interest = results[results['NeuronID'] == neuron]

In [17]:
import os
from analyses.enums.monkey_names import get_monkeys_by_default_order

monkey_group = "Zombies"
subject_monkey_index = 6
monkey_list = get_monkeys_by_default_order(monkey_group)
base_dir = '/home/connorlab/Documents/GitHub/Julie/social_data/zombies_social_data/'
behavior_files = {
    "AffiliationTo": "zombies_feature_df_affiliation.xlsx",
    "AffiliationFrom": "zombies_feature_df_affiliation.xlsx",
    "SubmissionTo": "zombies_feature_df_submission.xlsx",
    "SubmissionFrom": "zombies_feature_df_submission.xlsx",
    "AgonismTo": "zombies_feature_df_agonism.xlsx",
    "AgonismFrom": "zombies_feature_df_agonism.xlsx",
}

behavior_matrices = {
    name: pd.read_excel(os.path.join(base_dir, fname)).iloc[:, 1:].to_numpy().T if 'From' in name
    else pd.read_excel(os.path.join(base_dir, fname)).iloc[:, 1:].to_numpy()
    for name, fname in behavior_files.items()
}

In [28]:
neurons = results['NeuronID'].unique()

In [29]:
neurons = results['NeuronID'].unique()
for neuron in neurons:
    neuron_of_interest = results[results['NeuronID'] == neuron]

    # Get unique (behavior, source) pairs
    pairs = neuron_of_interest[['Behavior', 'Source_Monkey']].drop_duplicates()
    n = len(pairs)
    ncols = 3
    nrows = (n + ncols - 1) // ncols

    fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
    axs = axs.flatten()
    fig.suptitle(f'Neuron: {neuron}', fontsize=16, y=1.02)

    for idx, row in enumerate(pairs.itertuples(index=False)):
        behavior = row.Behavior
        source = row.Source_Monkey
        ax = axs[idx]

        source_idx = monkey_list.index(source)
        behavior_mat = behavior_matrices[behavior]

        # Mask to exclude subject and source
        mask = np.ones(len(monkey_list), dtype=bool)
        mask[subject_monkey_index] = False
        mask[source_idx] = False

        stimulus_monkeys = [m for i, m in enumerate(monkey_list) if mask[i]]
        behavioral_scores = behavior_mat[source_idx][mask]  # z-scored later
        behavioral_scores = (behavioral_scores - np.mean(behavioral_scores)) / np.std(behavioral_scores)

        df_subset = neuron_of_interest[
            (neuron_of_interest['Behavior'] == behavior) &
            (neuron_of_interest['Source_Monkey'] == source)
        ]
        r2 = df_subset['R-squared'].iloc[0] if not df_subset.empty else np.nan
        # Match by monkey name (stimulus)
        spike_rates = []
        labels = []
        for stim_monkey in stimulus_monkeys:
            row = df_subset[df_subset['StimulusMonkey'] == stim_monkey]
            if not row.empty:
                spike_rates.append(row['MeanSpikeRate'].values[0])
                labels.append(stim_monkey)
            else:
                spike_rates.append(np.nan)
                labels.append(stim_monkey)

        spike_rates = np.array(spike_rates)
        valid = ~np.isnan(spike_rates)
        x = behavioral_scores[valid]
        y = spike_rates[valid]
        labels = np.array(labels)[valid]

        # Plot
        for xi, yi, label in zip(x, y, labels):
            ax.scatter(xi, yi, color='blue')
            ax.text(xi, yi, label, fontsize=8, ha='right', va='bottom', alpha=0.6)

        if len(x) >= 2:
            coef = np.polyfit(x, y, 1)
            x_fit = np.linspace(min(x), max(x), 100)
            y_fit = coef[0] * x_fit + coef[1]
            ax.plot(x_fit, y_fit, color='black', linestyle='--')
            ax.text(0.05, 0.95, f'R²={r2:.2f}', transform=ax.transAxes,fontsize=10, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))


        ax.set_title(f'\n {behavior} {source}')
        ax.set_xlabel('Behavioral Score (z)')
        ax.set_ylabel('Mean Spike Rate')
        ax.grid(True)

    # Turn off unused subplots
    for j in range(idx + 1, len(axs)):
        axs[j].axis('off')
    save_folder = '/home/connorlab/Documents/GitHub/Julie/Cortana/analysis_results/neuron_profiling/'
    # plt.tight_layout()
    plt.savefig(save_folder + neuron + '.png')
    # plt.show()
    plt.close()